# Autômatos para Assistência Técnica de Impressoras
**Projeto acadêmico — Ciência da Computação, Noturno**

Felipe Ferreira - 2993620

Nicole Vieira - 2438577

Sthefany Alves - 2457820

Vitória Santos - 2392314

Igor Xavier - 2352643

In [1]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import re
from dataclasses import dataclass
from typing import List, Optional
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

print("Ambiente preparado. Python e ipywidgets carregados.")

Ambiente preparado. Python e ipywidgets carregados.


## R3 — Validadores de formatos e bateria de testes

Usaremos duas RegEx:
- Chamado: `CH` seguido de exatamente quatro dígitos.
- Impressora: `IMP-` seguido de exatamente três dígitos.

Cada bateria contém pelo menos cinco aceitos e cinco rejeitados, incluindo a string vazia. A coluna **Divergência** compara o resultado esperado com a RegEx.

In [ ]:
VALIDADORES = {
    "Número de chamado": {
        "regex": r"^CH[0-9]{4}$",
        "aceitos": ["CH1234", "CH0001", "CH9876", "CH4321", "CH9999"],
        "rejeitados": ["", "CH123", "CH12345", "1234", "CH12AB", "ch1234"]
    },
    "Código de impressora": {
        "regex": r"^IMP-[0-9]{3}$",
        "aceitos": ["IMP-123", "IMP-001", "IMP-999", "IMP-456", "IMP-007"],
        "rejeitados": ["", "IMP12", "IMP-12", "IMP-1234", "IMP-ABC", "imp-123"]
    }
}

def validar_formato(nome, valor):
    """Valida a string inteira com a RegEx associada ao formato."""
    return re.fullmatch(VALIDADORES[nome]["regex"].removeprefix("^").removesuffix("$"), valor) is not None

def executar_bateria(nome):
    cfg = VALIDADORES[nome]
    resultados = []
    for valor in cfg["aceitos"]:
        observado = validar_formato(nome, valor)
        resultados.append((valor if valor else "∅ (vazia)", "Aceitar", observado, observado is True))
    for valor in cfg["rejeitados"]:
        observado = validar_formato(nome, valor)
        resultados.append((valor if valor else "∅ (vazia)", "Rejeitar", observado, observado is False))
    return resultados

for nome, cfg in VALIDADORES.items():
    resultados = executar_bateria(nome)
    divergencias = [r for r in resultados if not r[3]]
    print(f"\n{nome} — RegEx: {cfg['regex']}")
    print(f"{'Entrada':<15} {'Esperado':<10} {'RegEx aceita?':<15} Divergência")
    for entrada, esperado, observado, ok in resultados:
        print(f"{entrada:<15} {esperado:<10} {str(observado):<15} {'Não' if ok else 'SIM'}")
    print(f"Total: {len(resultados)} testes | Divergências: {len(divergencias)}")

In [ ]:
# Interface interativa do R3
validador_dropdown = widgets.Dropdown(
    options=list(VALIDADORES.keys()), description="Formato:"
)
entrada_formato = widgets.Text(description="Entrada:", placeholder="Ex.: CH1234")
botao_validar = widgets.Button(description="Validar", button_style="primary")
botao_bateria = widgets.Button(description="Executar bateria")
saida_validacao = widgets.Output()

def ao_validar(_):
    with saida_validacao:
        clear_output()
        nome = validador_dropdown.value
        valor = entrada_formato.value
        ok = validar_formato(nome, valor)
        print(f"RegEx: {VALIDADORES[nome]['regex']}")
        print("VÁLIDO" if ok else "INVÁLIDO")

def ao_bateria(_):
    with saida_validacao:
        clear_output()
        nome = validador_dropdown.value
        resultados = executar_bateria(nome)
        for entrada, esperado, observado, ok in resultados:
            print(f"{entrada:<15} esperado={esperado:<8} RegEx={observado!s:<5} divergência={'não' if ok else 'SIM'}")
        print(f"\nDivergências: {sum(not r[3] for r in resultados)}")

botao_validar.on_click(ao_validar)
botao_bateria.on_click(ao_bateria)
display(widgets.VBox([
    validador_dropdown, entrada_formato,
    widgets.HBox([botao_validar, botao_bateria]), saida_validacao
]))

## R4 — Máquina de estados de negócio

Estados: `ABERTO`, `EM_ATENDIMENTO`, `EM_MANUTENCAO`, `AGUARDANDO_PECA`, `RESOLVIDO` e `FECHADO`.

Se um evento não for permitido, o sistema recusa a transição e informa os eventos permitidos no estado atual.

In [ ]:
ESTADOS = {
    "ABERTO", "EM_ATENDIMENTO", "EM_MANUTENCAO",
    "AGUARDANDO_PECA", "RESOLVIDO", "FECHADO"
}
TRANSICOES = {
    ("ABERTO", "INICIAR"): "EM_ATENDIMENTO",
    ("EM_ATENDIMENTO", "ENCAMINHAR_MANUTENCAO"): "EM_MANUTENCAO",
    ("EM_ATENDIMENTO", "RESOLVER"): "RESOLVIDO",
    ("EM_MANUTENCAO", "AGUARDAR_PECA"): "AGUARDANDO_PECA",
    ("AGUARDANDO_PECA", "PECA_RECEBIDA"): "EM_MANUTENCAO",
    ("EM_MANUTENCAO", "RESOLVER"): "RESOLVIDO",
    ("RESOLVIDO", "FECHAR"): "FECHADO",
    # Reabertura controlada para permitir correção após testes
    ("RESOLVIDO", "REABRIR"): "EM_ATENDIMENTO",
}
EVENTOS = sorted({evento for _, evento in TRANSICOES})

def eventos_permitidos(estado):
    return sorted(evento for (origem, evento), _ in TRANSICOES.items() if origem == estado)

def aplicar_evento(estado, evento):
    destino = TRANSICOES.get((estado, evento))
    if destino is None:
        permitidos = eventos_permitidos(estado)
        return estado, False, (
            f"Evento '{evento}' recusado no estado {estado}. "
            f"Eventos permitidos: {', '.join(permitidos) if permitidos else 'nenhum'}."
        )
    return destino, True, f"Transição válida: {estado} --{evento}--> {destino}"

def simular_fluxo(eventos, deve_ser_valido=True):
    estado = "ABERTO"
    historico = [estado]
    mensagens = []
    for evento in eventos:
        novo_estado, aceito, mensagem = aplicar_evento(estado, evento)
        mensagens.append(mensagem)
        if not aceito:
            if deve_ser_valido:
                return False, historico, mensagens
            return True, historico, mensagens
        estado = novo_estado
        historico.append(estado)
    sucesso = estado == "FECHADO" if deve_ser_valido else True
    return sucesso, historico, mensagens

FLUXOS = [
    ("Válido 1", ["INICIAR", "RESOLVER", "FECHAR"], True),
    ("Válido 2", ["INICIAR", "ENCAMINHAR_MANUTENCAO", "AGUARDAR_PECA",
                  "PECA_RECEBIDA", "RESOLVER", "FECHAR"], True),
    ("Inválido 1", ["FECHAR"], False),  # Não se fecha um chamado ABERTO
    ("Inválido 2", ["INICIAR", "AGUARDAR_PECA"], False),  # evento errado em atendimento
]

for nome, fluxo, esperado_valido in FLUXOS:
    estado = "ABERTO"
    historico = [estado]
    recusado = False
    mensagens = []
    for evento in fluxo:
        novo, ok, msg = aplicar_evento(estado, evento)
        mensagens.append(msg)
        if not ok:
            recusado = True
            break
        estado = novo
        historico.append(estado)
    observado_valido = (not recusado and estado == "FECHADO")
    print(f"\\n{nome}: {'VÁLIDO' if observado_valido else 'INVÁLIDO'}")
    print("Fluxo:", " -> ".join(historico))
    for msg in mensagens:
        print(" •", msg)
    assert observado_valido == esperado_valido, f"Falha no teste {nome}"
print("\\nTodos os quatro cenários foram verificados.")

In [ ]:
# Interface interativa do R4
estado_dropdown = widgets.Dropdown(options=sorted(ESTADOS), value="ABERTO", description="Estado:")
evento_dropdown = widgets.Dropdown(options=EVENTOS, description="Evento:")
botao_transicao = widgets.Button(description="Aplicar evento", button_style="primary")
botao_reset = widgets.Button(description="Reiniciar")
saida_estado = widgets.Output()

def atualizar_eventos(*_):
    permitidos = eventos_permitidos(estado_dropdown.value)
    opcoes = sorted(set(EVENTOS) | set(permitidos))
    evento_dropdown.options = opcoes
    if permitidos:
        evento_dropdown.value = permitidos[0]

def ao_aplicar(_):
    with saida_estado:
        clear_output()
        atual = estado_dropdown.value
        evento = evento_dropdown.value
        destino, ok, msg = aplicar_evento(atual, evento)
        print(msg)
        if ok:
            estado_dropdown.value = destino
        print("Estado atual:", estado_dropdown.value)
        print("Eventos permitidos:", ", ".join(eventos_permitidos(estado_dropdown.value)) or "nenhum")

def ao_reset(_):
    estado_dropdown.value = "ABERTO"
    with saida_estado:
        clear_output()
        print("Chamado reiniciado no estado ABERTO.")
        print("Eventos permitidos:", ", ".join(eventos_permitidos("ABERTO")))

estado_dropdown.observe(atualizar_eventos, names="value")
botao_transicao.on_click(ao_aplicar)
botao_reset.on_click(ao_reset)
atualizar_eventos()
display(widgets.VBox([
    estado_dropdown, evento_dropdown,
    widgets.HBox([botao_transicao, botao_reset]), saida_estado
]))

## R5 — Mini-scanner da linguagem de comandos

O scanner reconhece palavras reservadas, identificadores de chamado/impressora, inteiros, reais, operadores, pontuação e palavras inválidas. Espaços e comentários `# ...` são ignorados. A regra de **maximal munch** escolhe o token válido mais longo, por exemplo `==` antes de `=`, e `<=` antes de `<`.

Erros incluem linha, coluna e dicas relacionadas ao suporte técnico; o scanner avança e procura outros tokens depois de cada erro.

In [ ]:
@dataclass
class Token:
    tipo: str
    lexema: str
    linha: int
    coluna: int

@dataclass
class ErroLexico:
    caractere: str
    linha: int
    coluna: int
    dica: str

PALAVRAS_RESERVADAS = {
    "ABRIR", "CONSULTAR", "INICIAR", "MANUTENCAO",
    "RESOLVER", "FECHAR", "PRIORIDADE", "TEMPO"
}
PADROES_TOKENS = [
    ("CHAMADO", re.compile(r"CH[0-9]{4}")),
    ("IMPRESSORA", re.compile(r"IMP-[0-9]{3}")),
    ("REAL", re.compile(r"(?:[0-9]+\\.[0-9]+)")),
    ("INTEIRO", re.compile(r"[0-9]+")),
    ("IDENTIFICADOR", re.compile(r"[A-Za-z_][A-Za-z0-9_]*")),
    ("IGUAL_IGUAL", re.compile(r"==")),
    ("MENOR_IGUAL", re.compile(r"<=")),
    ("MAIOR_IGUAL", re.compile(r">=")),
    ("DIFERENTE", re.compile(r"!=")),
    ("IGUAL", re.compile(r"=")),
    ("MENOR", re.compile(r"<")),
    ("MAIOR", re.compile(r">")),
    ("MENOS", re.compile(r"-")),
    ("MAIS", re.compile(r"\\+")),
    ("MULTIPLICACAO", re.compile(r"\\*")),
    ("DIVISAO", re.compile(r"/")),
    ("ABRE_PARENTESE", re.compile(r"\\(")),
    ("FECHA_PARENTESE", re.compile(r"\\)")),
    ("VIRGULA", re.compile(r",")),
    ("PONTO_E_VIRGULA", re.compile(r";")),
]

def dica_para(caractere):
    if caractere == "@":
        return "Use ABRIR/CONSULTAR e um código como CH1234."
    if caractere == "$":
        return "Identificadores de impressora seguem o formato IMP-001."
    if caractere == ":":
        return "Separe os argumentos com espaços; exemplo: ABRIR CH1234 IMP-001."
    return "Revise o comando; use palavras reservadas e códigos CH1234 ou IMP-001."

def scanner(codigo):
    tokens, erros = [], []
    i, linha, coluna = 0, 1, 1
    while i < len(codigo):
        c = codigo[i]
        if c in " \\t\\r":
            i += 1
            coluna += 1
            continue
        if c == "\\n":
            i += 1
            linha += 1
            coluna = 1
            continue
        if c == "#":
            while i < len(codigo) and codigo[i] != "\\n":
                i += 1
                coluna += 1
            continue

        trecho = codigo[i:]
        candidatos = []
        for tipo, padrao in PADROES_TOKENS:
            m = padrao.match(trecho)
            if m:
                lexema = m.group(0)
                # Chamado/impressora devem ser reconhecidos antes de identificadores.
                candidatos.append((len(lexema), tipo, lexema))
        if candidatos:
            tamanho, tipo, lexema = max(candidatos, key=lambda item: item[0])
            if tipo == "IDENTIFICADOR" and lexema in PALAVRAS_RESERVADAS:
                tipo = "RESERVADA"
            tokens.append(Token(tipo, lexema, linha, coluna))
            i += tamanho
            coluna += tamanho
        else:
            erros.append(ErroLexico(c, linha, coluna, dica_para(c)))
            i += 1
            coluna += 1
    return tokens, erros

EXEMPLO_SCANNER = """# Abrir chamado de manutenção
ABRIR CH1234 IMP-001
PRIORIDADE CH1234 = 2
TEMPO CH1234 = 1.5
CONSULTAR CH1234
"""
tokens, erros = scanner(EXEMPLO_SCANNER)
print("TOKENS:")
for t in tokens:
    print(f"{t.tipo:<18} {t.lexema!r:<18} linha={t.linha}, coluna={t.coluna}")
print("\\nERROS:", erros or "nenhum")

In [ ]:
# Demonstração de maximal munch e recuperação após erros
amostra = "PRIORIDADE CH1234 == 2 <= 3 @ CONSULTAR CH1234 $"
tokens_demo, erros_demo = scanner(amostra)
print("Entrada:", amostra)
print("\\nTokens encontrados:")
for t in tokens_demo:
    print(f"{t.tipo:<18} {t.lexema!r} (L{t.linha}, C{t.coluna})")
print("\\nErros (o scanner continuou após cada erro):")
for e in erros_demo:
    print(f"Caractere {e.caractere!r} na linha {e.linha}, coluna {e.coluna}. Dica: {e.dica}")
assert any(t.tipo == "IGUAL_IGUAL" and t.lexema == "==" for t in tokens_demo)
assert any(t.lexema == "CONSULTAR" for t in tokens_demo)
assert len(erros_demo) == 2
print("\\nTeste de maximal munch e recuperação: OK")

In [ ]:
# Interface interativa do R5
entrada_codigo = widgets.Textarea(
    value="ABRIR CH1234 IMP-001\nPRIORIDADE CH1234 == 2 @\nCONSULTAR CH1234 # comentário",
    description="Código:", layout=widgets.Layout(width="100%", height="160px")
)
botao_scanner = widgets.Button(description="Executar scanner", button_style="primary")
saida_scanner = widgets.Output()

def ao_scanner(_):
    with saida_scanner:
        clear_output()
        toks, errs = scanner(entrada_codigo.value)
        print("TOKENS")
        for t in toks:
            print(f"{t.tipo:<18} {t.lexema!r:<18} linha={t.linha}, coluna={t.coluna}")
        print(f"\nTotal de tokens: {len(toks)}")
        print("\nERROS LÉXICOS")
        if not errs:
            print("Nenhum erro.")
        for e in errs:
            print(f"Linha {e.linha}, coluna {e.coluna}: caractere {e.caractere!r}. Dica: {e.dica}")
        print(f"Total de erros: {len(errs)}")

botao_scanner.on_click(ao_scanner)
display(widgets.VBox([entrada_codigo, botao_scanner, saida_scanner]))

## R6 — Painel integrado

As abas abaixo permitem demonstrar os validadores, a máquina de estados e o mini-scanner sem sair do notebook. Execute esta célula para abrir o painel.

In [2]:
aba_validadores = widgets.VBox([
    widgets.HTML("<h3>R3 — Validadores</h3>"),
    widgets.HTML("Use a interface de validadores na seção R3 acima.")
])
aba_estados = widgets.VBox([
    widgets.HTML("<h3>R4 — Máquina de estados</h3>"),
    widgets.HTML("Use a interface de estados na seção R4 acima.")
])
aba_scanner = widgets.VBox([
    widgets.HTML("<h3>R5 — Mini-scanner</h3>"),
    widgets.HTML("Use a interface do scanner na seção R5 acima.")
])
painel = widgets.Tab(children=[aba_validadores, aba_estados, aba_scanner])
painel.set_title(0, "Validadores")
painel.set_title(1, "Estados")
painel.set_title(2, "Scanner")
display(painel)